# Annual LULCC community fractions from cohort subtypes

This notebook calculates annual GDSTEM community fractions from LULCC cohort subtypes and their recorded areas.

1. Read longitude, latitude, year, cohort area, and subtype from the annual LULCC data in chunks.
2. Sum cohort areas by grid cell, year, and subtype in an on-disk SQLite database.
3. Map subtypes to communities 1–23. Report mixed, reserved, missing, and unmapped subtypes as unclassified.
4. Calculate each annual community fraction by dividing its summed area by the total retained cohort area in the same grid cell and year.
5. Calculate the mean, minimum, and maximum annual fraction for each community. Assign zero when a community is absent from an observed grid-year, and exclude years with no retained records for that grid cell.
6. Export community fraction statistics, unclassified subtypes, community coverage, grid coordinates, and a run summary.

## 1. Imports and configuration

Set the input path, output directories, inclusive years, and chunk size.
Reuse a database only after its CSV scan has completed. By default, stop before
fraction export if any retained area remains unclassified.


In [ ]:
import json
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd


# Configuration: review these settings before running.

# Set this to the LULCC input file verified for your server.
LULCC_FILE = Path('/quobyte/emoniergrp/TRENDY/input-files/global-input/gcb2025/lulcc-TRENDY-1700-2024.csv')

# Directory for Stage 1 output files.
# The subtype filename suffix separates these outputs from earlier classifications.
OUTPUT_DIR = Path('/quobyte/emoniergrp/luyy123/calibration_gridcell_selection_2002_2022_v2')
WORK_DIR = OUTPUT_DIR  # Optional: use a scratch directory with enough disk space and write permission.

# Inclusive year range for this calculation.
START_YEAR = 2002
END_YEAR = 2022
LULCC_CHUNKSIZE = 300_000  # Reduce to 50_000 if memory is limited.

# Keep False for the first scan. An existing database causes the run to stop.
# Set True only after this version has finished scanning the entire input CSV.
REUSE_COMPLETED_DATABASE = False

# Keep True to export unclassified combinations and stop before fraction export.
# The completed database is retained and can be reused after reviewing the mapping.
# False allows partial classification: unknown area stays in each denominator.
FAIL_ON_UNCLASSIFIED = True


## 2. Community mapping

Map `subtype` to a single community.


In [ ]:
# All 23 supported community names; some may be absent from the input data.
COMMUNITY_NAMES = {
    1: 'Ice', 2: 'Alpine Tundra and Polar Desert', 3: 'Moist and Wet Tundra',
    4: 'Boreal Forests', 5: 'Temperate Coniferous Forests',
    6: 'Temperate Deciduous Forests', 7: 'Temperate Broadleaved Evergreen Forest',
    8: 'Short Grassland', 9: 'Tall Grassland', 10: 'Mediterranean Shrubland',
    11: 'Xeric Shrubland', 12: 'Xeric Woodland',
    13: 'Tropical Evergreen Forests', 14: 'Tropical Deciduous Forests',
    15: 'Impervious Surfaces', 16: 'Turf Lawn', 17: 'Soybean', 18: 'Maize',
    19: 'Fruit tree', 20: 'Switchgrass', 21: 'Vegetable Farm',
    22: 'Pasture', 23: 'Rangeland',
}

# Input subtype code -> community number. Verify the server ECD version separately.
# Dictionary keys are LULCC subtype codes; values are community numbers 1-23.
MODEL_ATOMIC_VEG_TO_COMMUNITY = {
    1: 1, 
    2: 2, 
    3: 3, 
    4: 4, 
    6: 4, 
    7: 3,
    8: 5, 
    9: 6, 
    11: 7, 
    13: 6, 
    14: 9, 
    15: 6, 
    16: 9,
    17: 9, 
    18: 9, 
    19: 8, 
    20: 9, 
    21: 10, 
    22: 11, 
    23: 12,
    24: 13, 
    25: 14, 
    28: 13, 
    29: 9, 
    30: 14, 
    31: 9, 
    32: 13, 
    33: 11,
    47: 15, 
    48: 16,
    50: 17, 
    51: 18, 
    52: 19, 
    53: 20,
    54: 17, 
    55: 18, 
    56: 19, 
    57: 20,
    58: 18,
    60: 22, 
    61: 23,
}


# Mixed vegetation must already be split into atomic cohorts in the input.
# Codes 40-46 are mixed urban types and also require review if selected for classification.
NONATOMIC_VEG_CODES = {5, 10, 12, 26, 27, 40, 41, 42, 43, 44, 45, 46}

# Reserved local tcomm codes must not be interpreted as real ice area.
RESERVED_VEG_CODES = {34, 35, 36, 37, 38, 39, 49, 59}

# Processing constants for subtype-only annual area accumulation.
SCALE = 1_000_000
MISSING_CODE = -999
SCHEMA_VERSION = 'stage1_subtype_v1'
KEYS = ['lon_i', 'lat_i', 'year', 'subtype']
INPUT_FIELDS = ['lon', 'lat', 'year', 'cohort_area', 'subtype']


## 3. Output paths and input columns


In [ ]:
def output_paths():
    """Name the Stage 1 outputs for the selected year range."""
    tag = f'{START_YEAR}_{END_YEAR}_v1'
    return {
        # One row per observed grid/community pair: fraction statistics,
        # observed-year counts, and mean area statistics (11 columns).
        'fractions': OUTPUT_DIR / f'01_community_fraction_{tag}.csv',

        # Only unclassified subtypes, with reasons, cohort row counts,
        # and area summed across all retained grids and years.
        'unclassified': OUTPUT_DIR / f'01_unclassified_{tag}.csv',

        # All 23 communities, including absent ones: grid counts,
        # grid-year counts, and area summed across retained years.
        'coverage': OUTPUT_DIR / f'01_class_coverage_{tag}.csv',

        # All retained grid coordinates and the number of observed years.
        'grid': OUTPUT_DIR / f'01_grid_coordinates_{tag}.csv',

        # Input identity, output paths, unclassified totals, and run status.
        'manifest': OUTPUT_DIR / f'01_run_summary_{tag}.json',

        # On-disk intermediate tables and scan metadata for database reuse.
        'db': WORK_DIR / f'01_work_lulcc_{tag}.sqlite',
    }


def input_columns():
    """Read the LULCC file using fixed zero-based column positions."""
    positions = {
        0: 'lon',
        1: 'lat',
        3: 'year',
        7: 'cohort_area',
        10: 'subtype',
    }
    return None, list(positions), positions


## 4. Assign a community from subtype

Known single-community subtypes map to their configured
community; mixed, reserved, missing, and unmapped subtypes receive community 0
and a reason for review. No fixed mixture percentages are applied.


In [ ]:
def classify_record(subtype):
    """Return the community and any classification issue for one input subtype."""
    if subtype == MISSING_CODE:
        return 0, 'Missing subtype'
    if subtype in NONATOMIC_VEG_CODES:
        return 0, 'Mixed subtype requires split cohorts'
    if subtype in RESERVED_VEG_CODES:
        return 0, 'Reserved subtype is not a valid community'
    community = MODEL_ATOMIC_VEG_TO_COMMUNITY.get(subtype, 0)
    if community not in COMMUNITY_NAMES:
        return 0, 'Subtype has no community mapping'
    return community, ''


## 5. Create or reuse the disk database

Store actual cohort area by grid, year, and subtype. Reuse requires a completed
scan with the same schema, input path, size, modification time, and year range.
These metadata are not a content checksum. Start with a fresh subtype-only
database; databases from the previous classification are not reused.


In [ ]:
def source_signature():
    """Record input identity and years to validate reuse of a completed database."""
    stat = LULCC_FILE.stat()
    return {'path': str(LULCC_FILE.resolve()), 'bytes': stat.st_size,
            'mtime_ns': stat.st_mtime_ns, 'start_year': START_YEAR, 'end_year': END_YEAR}


def meta_set(conn, key, value):
    """Save one metadata value as JSON text in the database."""
    json_value = json.dumps(value, ensure_ascii=False)
    conn.execute('INSERT OR REPLACE INTO metadata VALUES (?,?)', (key, json_value))


def meta_get(conn, key):
    """Read one metadata value; return None if the key is absent."""
    row = conn.execute('SELECT value FROM metadata WHERE key=?', (key,)).fetchone()
    if row is None:
        return None
    return json.loads(row[0])


def connect_db(path, reuse):
    """Create or validate a disk database containing grid-year-subtype area sums.
    
    The stored subtype sums allow remapping without rescanning the CSV."""
    conn = sqlite3.connect(path)
    # SQLite settings: keep temporary tables on disk and use about 32 MB of cache.
    # Retain these settings when the input is too large to hold in memory.
    conn.execute('PRAGMA journal_mode=DELETE')
    conn.execute('PRAGMA synchronous=NORMAL')
    conn.execute('PRAGMA temp_store=FILE')
    conn.execute('PRAGMA cache_size=-32000')
    if reuse:
        try:
            ok = (meta_get(conn, 'schema') == SCHEMA_VERSION and
                  meta_get(conn, 'scan_complete') is True and
                  meta_get(conn, 'source') == source_signature())
        except sqlite3.Error:
            ok = False
        if not ok:
            conn.close()
            raise ValueError('Cannot reuse database: schema, completed-scan flag, input file, or year range does not match.')
        return conn
    conn.execute('CREATE TABLE metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL)')
    conn.execute('''CREATE TABLE cohort_year (
        lon_i INTEGER NOT NULL, lat_i INTEGER NOT NULL, year INTEGER NOT NULL,
        subtype INTEGER NOT NULL,
        area REAL NOT NULL, n_rows INTEGER NOT NULL,
        PRIMARY KEY(lon_i,lat_i,year,subtype)
    ) WITHOUT ROWID''')
    meta_set(conn, 'schema', SCHEMA_VERSION)
    meta_set(conn, 'scan_complete', False)
    meta_set(conn, 'source', source_signature())
    conn.commit()
    return conn


## 6. Read and accumulate annual cohort areas

Select years and retain valid positive cohort area. Sum repeated grid-year-subtype
keys within and across chunks using the input area values without percentage
multipliers. Missing subtype values remain in the area denominator and are
reported during classification. Only subtype codes need integer validation.


In [ ]:
def ingest(conn):
    """Scan the CSV in chunks and store selected years with valid positive areas.
    
    Missing classification codes do not remove otherwise valid area."""
    header, usecols, rename = input_columns()
    print('Reading headerless LULCC; column map:', rename, flush=True)
    reader = pd.read_csv(LULCC_FILE, header=header, usecols=usecols,
                         chunksize=LULCC_CHUNKSIZE, skipinitialspace=True,
                         low_memory=False, encoding='utf-8-sig', on_bad_lines='error')
    # Repeated grid/year/subtype keys add their area and row count to the stored sums.
    query = '''INSERT INTO cohort_year VALUES (?,?,?,?,?,?)
               ON CONFLICT(lon_i,lat_i,year,subtype)
               DO UPDATE SET area=area+excluded.area, n_rows=n_rows+excluded.n_rows'''
    raw_rows = kept_rows = 0
    start = time.monotonic()
    for i, chunk in enumerate(reader, 1):
        chunk = chunk.rename(columns=rename)
        raw_rows += len(chunk)
        for field in INPUT_FIELDS:
            chunk[field] = pd.to_numeric(chunk[field], errors='coerce')
        # Retain valid positive areas even when classification fields are missing.
        keep = (chunk.year.between(START_YEAR, END_YEAR) &
                np.isfinite(chunk.lon) & np.isfinite(chunk.lat) &
                chunk.lat.between(-90, 90) & np.isfinite(chunk.cohort_area) &
                (chunk.cohort_area > 0))
        chunk = chunk.loc[keep].copy()
        if chunk.empty:
            if i % 100 == 0:
                print(f'chunks={i}, raw={raw_rows:,}, retained={kept_rows:,}', flush=True)
            continue
        if not chunk.year.eq(np.floor(chunk.year)).all():
            raise ValueError('The selected records contain noninteger years.')
        for field in ('subtype',):
            finite = np.isfinite(chunk[field])
            bad = finite & chunk[field].ne(np.floor(chunk[field]))
            if bad.any():
                raise ValueError(f'{field} contains noninteger codes: {chunk.loc[bad, field].head().tolist()}')
            # Preserve missing subtypes as a sentinel without dropping their area.
            chunk.loc[~finite, field] = MISSING_CODE
            # Check the range before converting to int32 to avoid integer overflow.
            if ((chunk[field] < np.iinfo(np.int32).min) |
                    (chunk[field] > np.iinfo(np.int32).max)).any():
                raise ValueError(f'{field} contains codes outside the int32 range.')
            chunk[field] = chunk[field].astype('int32')
        lon = ((chunk.lon.to_numpy() + 180.0) % 360.0) - 180.0
        chunk['lon_i'] = np.rint(lon * SCALE).astype('int32')
        chunk['lat_i'] = np.rint(chunk.lat.to_numpy() * SCALE).astype('int32')
        chunk['year'] = chunk.year.astype('int32')
        # Sum within each chunk, then across chunks in SQLite; retain original cohort row counts.
        grouped = chunk.groupby(KEYS, as_index=False, sort=False).agg(
            area=('cohort_area', 'sum'), n_rows=('cohort_area', 'size'))
        rows = grouped[KEYS + ['area', 'n_rows']].itertuples(index=False, name=None)
        with conn:
            conn.executemany(query, rows)
        kept_rows += len(chunk)
        if i % 20 == 0:
            print(f'chunks={i}, raw={raw_rows:,}, {START_YEAR}-{END_YEAR}={kept_rows:,}, '
                  f'elapsed={time.monotonic()-start:.0f}s', flush=True)
    if not kept_rows:
        raise ValueError('No records passed the year and area filters.')
    meta_set(conn, 'raw_rows', raw_rows)
    meta_set(conn, 'kept_rows', kept_rows)
    meta_set(conn, 'scan_complete', True)
    conn.commit()
    print('Completed LULCC scan, retained rows:', f'{kept_rows:,}', flush=True)


## 7. Export unclassified subtypes

Map each distinct subtype once. Export subtype, community 0, reason, original
cohort row count, and area summed across retained grids and years. An empty
unclassified CSV still has column headers. By default, unclassified area stops
the run before fraction export.


In [ ]:
def export_query(conn, query, path, names=False):
    """Write SQL query results to CSV in chunks, including an empty-file header.
    
    Set names=True to add the community_name column."""
    first = True
    for part in pd.read_sql_query(query, conn, chunksize=75_000):
        if names:
            part['community_name'] = part.community_type.map(COMMUNITY_NAMES)
        part.to_csv(path, index=False, mode='w' if first else 'a', header=first)
        first = False
    if first:
        # If pandas yields no chunks, use LIMIT 0 to obtain and write the column names.
        pd.read_sql_query('SELECT * FROM (' + query.rstrip(';') + ') LIMIT 0', conn).to_csv(path, index=False)


def build_classification(conn, paths):
    """Map distinct subtypes and export those without an assigned community."""
    conn.execute('DROP TABLE IF EXISTS mapping')
    conn.execute('''CREATE TABLE mapping (
        subtype INTEGER PRIMARY KEY,
        community_type INTEGER, reason TEXT
    ) WITHOUT ROWID''')

    subtypes = conn.execute('SELECT DISTINCT subtype FROM cohort_year').fetchall()
    rows = []
    for (subtype,) in subtypes:
        community, reason = classify_record(subtype)
        rows.append((subtype, community, reason))
    conn.executemany('INSERT INTO mapping VALUES (?,?,?)', rows)
    conn.commit()

    unknown = pd.read_sql_query('''
        SELECT c.subtype,m.community_type,m.reason,
            SUM(c.n_rows) AS n_cohort_rows,
            SUM(c.area) AS total_cohort_area_across_years
        FROM cohort_year c JOIN mapping m USING(subtype)
        WHERE m.community_type=0
        GROUP BY c.subtype
        ORDER BY c.subtype
    ''', conn)
    unknown.to_csv(paths['unclassified'], index=False)
    unknown_rows = int(unknown['n_cohort_rows'].sum())
    unknown_area = float(unknown['total_cohort_area_across_years'].sum())
    print('Unclassified cohort rows:', unknown_rows, flush=True)
    return unknown_rows, unknown_area


## 8. Calculate annual fractions and multi-year statistics

For grid $g$, observed year $y$, and community $c$, sum actual input cohort areas:

$$A_{g,y}=\sum_{i\in(g,y)}a_i,\qquad
A_{g,y,c}=\sum_{i\in(g,y),\,M(\mathrm{subtype}_i)=c}a_i.$$

Compute each annual fraction before averaging:

$$f_{g,y,c}=\frac{A_{g,y,c}}{A_{g,y}},\qquad
\bar f_{g,c}=\frac{1}{|Y_g|}\sum_{y\in Y_g}f_{g,y,c}.$$

Minimum and maximum are taken over these annual fractions. A community absent
in an observed grid-year contributes zero. A year without retained grid records
is excluded. The multi-year mean is not a ratio of multi-year area sums.
Unclassified area remains in the denominator if partial classification is allowed.
Only grid/community pairs present at least once appear in the fraction table.

`mean_present_community_area` averages area over community-present years only.
`mean_total_cohort_area` averages total area over all observed grid-years.
The coverage summary always lists communities 1-23.


In [ ]:
def aggregate(conn, paths):
    """Calculate annual fractions and multi-year statistics with the original formulas."""
    # Remove previous annual totals before recalculating.
    conn.execute('''
        DROP TABLE IF EXISTS grid_year
    ''')
    # Remove previous annual community areas before recalculating.
    conn.execute('''
        DROP TABLE IF EXISTS community_year
    ''')
    # Denominator: sum all retained input cohort areas for each grid-year.
    conn.execute('''
        CREATE TABLE grid_year AS
          SELECT lon_i,lat_i,year,SUM(area) AS total_area
          FROM cohort_year GROUP BY lon_i,lat_i,year
    ''')
    # Index grid-year keys to speed up the annual-area join.
    conn.execute('''
        CREATE UNIQUE INDEX idx_grid_year ON grid_year(lon_i,lat_i,year)
    ''')
    # Numerator: sum input cohort areas assigned by subtype to each community.
    conn.execute('''
        CREATE TABLE community_year AS
          SELECT c.lon_i,c.lat_i,c.year,m.community_type,SUM(c.area) AS community_area
          FROM cohort_year c JOIN mapping m USING(subtype)
          WHERE m.community_type BETWEEN 1 AND 23
          GROUP BY c.lon_i,c.lat_i,c.year,m.community_type
    ''')
    # Index community grid-year keys to speed up aggregation.
    conn.execute('''
        CREATE INDEX idx_community_year ON community_year(lon_i,lat_i,year)
    ''')
    # Annual denominators contain all valid cohort area, including unclassified area.
    # Community absence counts as zero; years with no grid records are excluded.
    # SQL steps: annual fractions -> community statistics -> observed-year counts.
    # The final mean is sum(annual fraction) / n_years_grid, not a ratio of area sums.
    query = '''WITH yearly_fraction AS (
        SELECT c.lon_i,c.lat_i,c.year,c.community_type,c.community_area,
          c.community_area/g.total_area AS fraction
        FROM community_year c JOIN grid_year g USING(lon_i,lat_i,year)
        WHERE g.total_area>0),
      group_stats AS (
        SELECT lon_i,lat_i,community_type,SUM(fraction) AS sum_fraction,
          MIN(fraction) AS min_present_fraction,MAX(fraction) AS max_fraction,
          COUNT(*) AS n_years_community,AVG(community_area) AS mean_present_community_area
        FROM yearly_fraction GROUP BY lon_i,lat_i,community_type),
      grid_stats AS (
        SELECT lon_i,lat_i,COUNT(*) AS n_years_grid,AVG(total_area) AS mean_total_cohort_area
        FROM grid_year WHERE total_area>0 GROUP BY lon_i,lat_i)
      SELECT g.lon_i,g.lat_i,g.community_type,
        g.sum_fraction/s.n_years_grid AS mean_community_fraction,
        CASE WHEN g.n_years_community<s.n_years_grid THEN 0
             ELSE g.min_present_fraction END AS min_community_fraction,
        g.max_fraction AS max_community_fraction,g.n_years_community,s.n_years_grid,
        g.mean_present_community_area,s.mean_total_cohort_area
      FROM group_stats g JOIN grid_stats s USING(lon_i,lat_i)
      ORDER BY g.community_type,g.lon_i,g.lat_i'''
    columns = ['lon', 'lat', 'community_type', 'community_name',
               'mean_community_fraction', 'min_community_fraction', 'max_community_fraction',
               'n_years_community', 'n_years_grid', 'mean_present_community_area',
               'mean_total_cohort_area']
    first = True
    for part in pd.read_sql_query(query, conn, chunksize=75_000):
        part['lon'] = part.pop('lon_i') / SCALE
        part['lat'] = part.pop('lat_i') / SCALE
        part['community_name'] = part.community_type.map(COMMUNITY_NAMES)
        part[columns].to_csv(paths['fractions'], index=False,
                             mode='w' if first else 'a', header=first)
        first = False
    if first:
        pd.DataFrame(columns=columns).to_csv(paths['fractions'], index=False)
        raise ValueError('No communities could be classified; review the community mapping.')
    # Always list communities 1-23; absent communities have zero observed coverage.
    coverage = pd.DataFrame({'community_type': range(1, 24)})
    stats = pd.read_sql_query('''SELECT community_type,COUNT(*) AS n_grid_years,
        SUM(community_area) AS total_cohort_area_across_years,
        COUNT(DISTINCT printf('%d,%d',lon_i,lat_i)) AS n_grids
        FROM community_year GROUP BY community_type''', conn)
    coverage = coverage.merge(stats, on='community_type', how='left').fillna(0)
    coverage['n_grid_years'] = coverage.n_grid_years.astype('int64')
    coverage['n_grids'] = coverage.n_grids.astype('int64')
    coverage['community_name'] = coverage.community_type.map(COMMUNITY_NAMES)
    coverage = coverage[['community_type', 'community_name', 'n_grids', 'n_grid_years',
                         'total_cohort_area_across_years']]
    coverage.to_csv(paths['coverage'], index=False)
    # Export all retained grids and their observed-year counts.
    export_query(conn, f'''SELECT lon_i*1.0/{SCALE} AS lon,lat_i*1.0/{SCALE} AS lat,
                 COUNT(*) AS n_years_grid FROM grid_year
                 GROUP BY lon_i,lat_i ORDER BY lat_i,lon_i''', paths['grid'])
    print(coverage.to_string(index=False), flush=True)


## 9. Run the workflow

Run the preceding cells in order, then run this cell. It scans or reuses the
subtype-only database, maps subtypes, and exports annual-fraction statistics and
summaries. If classification stops, review the unclassified subtypes and input
preprocessing before remapping. A completed scan can be reused after review.


In [ ]:
def main():
    """Run Stage 1 and save a compact record of completion or failure."""
    if START_YEAR > END_YEAR or LULCC_CHUNKSIZE <= 0:
        raise ValueError('Invalid year range or CSV chunk size.')
    if not LULCC_FILE.is_file():
        raise FileNotFoundError(LULCC_FILE)

    paths = output_paths()
    exists = paths['db'].exists()
    if exists and not REUSE_COMPLETED_DATABASE:
        raise FileExistsError(
            f"Database already exists: {paths['db']}. "
            'Use REUSE_COMPLETED_DATABASE=True only for a completed compatible scan.'
        )
    if REUSE_COMPLETED_DATABASE and not exists:
        raise FileNotFoundError(paths['db'])

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    conn = connect_db(paths['db'], reuse=exists)
    manifest = {
        'status': 'running',
        'source': source_signature(),
        'classification_definition': 'subtype_only',
        'paths': {},
    }
    for name, path in paths.items():
        manifest['paths'][name] = str(path)

    try:
        if not exists:
            ingest(conn)
        unknown_rows, unknown_area = build_classification(conn, paths)
        manifest['unclassified_cohort_rows'] = unknown_rows
        manifest['unclassified_total_cohort_area_across_years'] = unknown_area
        if unknown_rows and FAIL_ON_UNCLASSIFIED:
            manifest['status'] = 'needs_classification_review'
            raise ValueError(f"Review unclassified combinations: {paths['unclassified']}")

        aggregate(conn, paths)
        manifest['status'] = 'complete'
        if unknown_rows:
            manifest['status'] = 'complete_with_unclassified_area'
        print('Saved community fractions:', paths['fractions'], flush=True)
    except Exception as exc:
        if manifest['status'] == 'running':
            manifest['status'] = 'failed'
        manifest['error'] = str(exc)
        raise
    finally:
        conn.close()
        paths['manifest'].write_text(json.dumps(manifest, indent=2), encoding='utf-8')
        print('Run summary:', paths['manifest'], flush=True)


main()


## Pipeline outputs

| Output key | Contents |
| --- | --- |
| `fractions` | Coordinates, community identity, mean/minimum/maximum annual fractions, year counts, and area statistics; 11 columns |
| `unclassified` | Unclassified subtypes, reasons, cohort row counts, and area summed across years |
| `coverage` | All 23 communities with grid counts, grid-year counts, and area summed across years |
| `grid` | Retained grid coordinates and observed-year counts |
| `manifest` | Input identity, output paths, classification totals, run status, and any error |
| `db` | Intermediate SQLite tables and scan-completion metadata |

Only the unclassified CSV and run summary are guaranteed to be written after a
classification stop; fraction and coverage exports require the run to reach aggregation.
The database is kept for reuse. Files from previous runs may still exist.

